# local-db-test: local database demo with 

In [26]:
import logging
import sys
import subprocess
from pathlib import Path

In [27]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

In [28]:
from dve.config import conf

In [29]:
logging.basicConfig(stream=sys.stderr, level=logging.DEBUG)

log = logging.getLogger(__name__)

In [30]:
cnf = conf.get_config()

In [31]:
local_uri = conf.get_config().db("demo").uri()

In [32]:
print(f"local_uri: {conf.get_config().db('demo').dump()}")
print(f"local_dsc: {conf.get_config().db('demo').dump(full=True)}")

local_uri: sqlite+pysqlite:///data/ext/test/demo/demo.db
local_dsc: {
    "alias": "demo_lt"
}


In [33]:
! pwd

/root/work/vb/dve-sample-r/notebooks/sys-test


In [34]:
# jupyter current dir ("./notebooks") adjustment
local_uri = local_uri.replace("///", "///../../")

In [35]:
print(local_uri)

sqlite+pysqlite:///../../data/ext/test/demo/demo.db


In [36]:
def sql_bind(engine):
    def sql_exec(qry:str):
        # engine.begin() automatically commits the transaction on success
        with engine.begin() as con:
            rs = con.execute(text(qry))
        
        # safely check if the result set is iterable (e.g., SELECT queries)
        if rs.returns_rows:
            for row in rs:
                print(row)
        else:
            # For CREATE/INSERT/UPDATE, just print the result status
            print(f"Executed successfully. Rows affected: {rs.rowcount}")
    return sql_exec        


def drop_database(uri: str):
    # 0. Parse the actual file path from the SQLAlchemy URI automatically
    db_file_path = make_url(local_uri).database

    if db_file_path is None:
        return None
        
    db_path = Path(db_file_path)
    
    # 1. Ensure parent directory exists (creates 'dbms' and any missing parent directories)
    db_path.parent.mkdir(parents=True, exist_ok=True)
        
    # 2. Delete main database file and temporary WAL/SHM journal files if present
    db_path.unlink(missing_ok=True)
    Path(f"{db_path}-wal").unlink(missing_ok=True)
    Path(f"{db_path}-shm").unlink(missing_ok=True)
    
    # 3. Ensure the parent directory exists
    db_path.parent.mkdir(parents=True, exist_ok=True)
    return db_path


In [37]:
db_path = drop_database(local_uri)

In [38]:
engine = create_engine(local_uri, echo=True)

In [39]:
sql_exec = sql_bind(engine)

In [40]:
sql_exec("SELECT current_timestamp")

2026-10-01 19:07:49,219 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:49,220 INFO sqlalchemy.engine.Engine SELECT current_timestamp


INFO:sqlalchemy.engine.Engine:SELECT current_timestamp


2026-10-01 19:07:49,220 INFO sqlalchemy.engine.Engine [generated in 0.00068s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00068s] ()


2026-10-01 19:07:49,221 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('2026-10-01 17:07:49',)


In [41]:
schema_sql = """

SELECT 
  m.name as table_name, 
  p.name as column_name
FROM 
  sqlite_master AS m
JOIN 
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table' 
ORDER BY 
  m.name, 
  p.cid

"""

In [42]:
sql_exec(schema_sql)

2026-10-01 19:07:52,356 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:52,357 INFO sqlalchemy.engine.Engine 

SELECT 
  m.name as table_name, 
  p.name as column_name
FROM 
  sqlite_master AS m
JOIN 
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table' 
ORDER BY 
  m.name, 
  p.cid




INFO:sqlalchemy.engine.Engine:

SELECT 
  m.name as table_name, 
  p.name as column_name
FROM 
  sqlite_master AS m
JOIN 
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table' 
ORDER BY 
  m.name, 
  p.cid




2026-10-01 19:07:52,357 INFO sqlalchemy.engine.Engine [generated in 0.00056s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00056s] ()


2026-10-01 19:07:52,358 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


In [43]:
create_sql = """

CREATE TABLE IF NOT EXISTS demo_table (
	column1 text PRIMARY KEY,
   	column2 text NOT NULL,
	column3 integer DEFAULT 0
);

"""

insert_sql = """

INSERT INTO 'demo_table' ('column1', 'column2') VALUES
  ('a', 'data1'),
  ('b', 'data2'),
  ('c', 'data1'),
  ('d', 'data3'),
  ('e', 'data3');
  
"""

update_sql = """

UPDATE 'demo_table' SET
  column3 = column3 + 1
 WHERE 
   column2 in ('data2', 'data3');
  
"""

select_sql = """

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;

"""

group_sql = """

SELECT column2, sum(column3) as tot3 FROM 'demo_table'
GROUp BY column2 HAVING tot3 > 0
ORDER BY tot3 DESC, column2;

"""

In [44]:
sql_exec(create_sql)

2026-10-01 19:07:55,408 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:55,409 INFO sqlalchemy.engine.Engine 

CREATE TABLE IF NOT EXISTS demo_table (
	column1 text PRIMARY KEY,
   	column2 text NOT NULL,
	column3 integer DEFAULT 0
);




INFO:sqlalchemy.engine.Engine:

CREATE TABLE IF NOT EXISTS demo_table (
	column1 text PRIMARY KEY,
   	column2 text NOT NULL,
	column3 integer DEFAULT 0
);




2026-10-01 19:07:55,410 INFO sqlalchemy.engine.Engine [generated in 0.00052s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00052s] ()


2026-10-01 19:07:55,444 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Executed successfully. Rows affected: -1


In [45]:
sql_exec(schema_sql)

2026-10-01 19:07:56,298 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:56,299 INFO sqlalchemy.engine.Engine 

SELECT 
  m.name as table_name, 
  p.name as column_name
FROM 
  sqlite_master AS m
JOIN 
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table' 
ORDER BY 
  m.name, 
  p.cid




INFO:sqlalchemy.engine.Engine:

SELECT 
  m.name as table_name, 
  p.name as column_name
FROM 
  sqlite_master AS m
JOIN 
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table' 
ORDER BY 
  m.name, 
  p.cid




2026-10-01 19:07:56,299 INFO sqlalchemy.engine.Engine [cached since 3.943s ago] ()


INFO:sqlalchemy.engine.Engine:[cached since 3.943s ago] ()


2026-10-01 19:07:56,300 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('demo_table', 'column1')
('demo_table', 'column2')
('demo_table', 'column3')


In [46]:
sql_exec(insert_sql)

2026-10-01 19:07:57,168 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:57,169 INFO sqlalchemy.engine.Engine 

INSERT INTO 'demo_table' ('column1', 'column2') VALUES
  ('a', 'data1'),
  ('b', 'data2'),
  ('c', 'data1'),
  ('d', 'data3'),
  ('e', 'data3');




INFO:sqlalchemy.engine.Engine:

INSERT INTO 'demo_table' ('column1', 'column2') VALUES
  ('a', 'data1'),
  ('b', 'data2'),
  ('c', 'data1'),
  ('d', 'data3'),
  ('e', 'data3');




2026-10-01 19:07:57,170 INFO sqlalchemy.engine.Engine [generated in 0.00051s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00051s] ()


2026-10-01 19:07:57,171 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Executed successfully. Rows affected: 5


In [47]:
sql_exec(select_sql)

2026-10-01 19:07:57,942 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:57,943 INFO sqlalchemy.engine.Engine 

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




INFO:sqlalchemy.engine.Engine:

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




2026-10-01 19:07:57,944 INFO sqlalchemy.engine.Engine [generated in 0.00066s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00066s] ()


2026-10-01 19:07:57,945 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('c', 'data1', 0)
('a', 'data1', 0)
('b', 'data2', 0)
('e', 'data3', 0)
('d', 'data3', 0)


In [48]:
sql_exec(update_sql)
sql_exec(update_sql)
sql_exec(update_sql)

2026-10-01 19:07:58,678 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:58,679 INFO sqlalchemy.engine.Engine 

UPDATE 'demo_table' SET
  column3 = column3 + 1
 WHERE 
   column2 in ('data2', 'data3');




INFO:sqlalchemy.engine.Engine:

UPDATE 'demo_table' SET
  column3 = column3 + 1
 WHERE 
   column2 in ('data2', 'data3');




2026-10-01 19:07:58,680 INFO sqlalchemy.engine.Engine [generated in 0.00069s] ()


...

2026-10-01 19:07:58,736 INFO sqlalchemy.engine.Engine [cached since 0.05706s ago] ()


INFO:sqlalchemy.engine.Engine:[cached since 0.05706s ago] ()


2026-10-01 19:07:58,737 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Executed successfully. Rows affected: 3


In [49]:
sql_exec(select_sql)

2026-10-01 19:07:59,490 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:07:59,491 INFO sqlalchemy.engine.Engine 

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




INFO:sqlalchemy.engine.Engine:

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




2026-10-01 19:07:59,492 INFO sqlalchemy.engine.Engine [cached since 1.548s ago] ()


INFO:sqlalchemy.engine.Engine:[cached since 1.548s ago] ()


2026-10-01 19:07:59,492 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('c', 'data1', 0)
('a', 'data1', 0)
('b', 'data2', 3)
('e', 'data3', 3)
('d', 'data3', 3)


In [50]:
sql_exec(group_sql)

2026-10-01 19:08:00,303 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-01 19:08:00,304 INFO sqlalchemy.engine.Engine 

SELECT column2, sum(column3) as tot3 FROM 'demo_table'
GROUp BY column2 HAVING tot3 > 0
ORDER BY tot3 DESC, column2;




INFO:sqlalchemy.engine.Engine:

SELECT column2, sum(column3) as tot3 FROM 'demo_table'
GROUp BY column2 HAVING tot3 > 0
ORDER BY tot3 DESC, column2;




2026-10-01 19:08:00,304 INFO sqlalchemy.engine.Engine [generated in 0.00050s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00050s] ()


2026-10-01 19:08:00,305 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('data3', 6)
('data2', 3)


In [51]:
subprocess.run(["ls", "-l", db_path.parent]) 

total 12
-rw-r--r-- 1 root root 12288 Oct  1 19:07 demo.db


CompletedProcess(args=['ls', '-l', PosixPath('../../data/ext/test/demo')], returncode=0)